GROUP 1 PROJECT DATA ANALYSIS 

## API connection ##

**Source:**  SMARD / Bundesnetzagentur -> ENTSO-E

As on SMARD at this moment is only possible to download data, and not to access it through an API and on webpage is written this: "SMARD receives the data directly from the European Network of Transmission System Operators for Electricity (ENTSO-E). Only data verified by the Bundesnetzagentur is published on SMARD.", ENTSO-E API is used instead.


Owner: Darija


In [27]:
from entsoe import EntsoePandasClient
import pandas as pd
#DL the module entsoe 

client = EntsoePandasClient(
    api_key='5bcf63bf-87d7-4aa5-8bfb-040cbc4c4af0'
)


##This is Kilian token 
start = pd.Timestamp(
    '2025-01-01',
    tz='Europe/Brussels'
)

end = pd.Timestamp(
    '2025-03-01',
    tz='Europe/Brussels'
)

es = client.query_day_ahead_prices(
    'ES',
    start=start,
    end=end
)

fr = client.query_day_ahead_prices(
    'FR',
    start=start,
    end=end
)

df = pd.DataFrame({
    'Spain': es,
    'France': fr
})

print(df.head())

                            Spain  France
2024-12-31 23:00:00+00:00  134.49   12.36
2025-01-01 00:00:00+00:00  131.59   18.92
2025-01-01 01:00:00+00:00  131.49   16.66
2025-01-01 02:00:00+00:00  131.42   13.10
2025-01-01 03:00:00+00:00  120.49    5.90


In [ ]:
##VOLATILITY ANALYSIS

# --- 3. Volatility Calculation ---
def calculate_volatility(df, country):
    # Global volatility
    volatility = {
        "std_dev": df[country].std(),
        "mean": df[country].mean(),
        "min": df[country].min(),
        "max": df[country].max(),
        "range": df[country].max() - df[country].min(),
        "coef_variation": (df[country].std() / df[country].mean()) * 100,
    }

    # Hourly volatility
    hourly_volatility = df.groupby("hour")[country].agg(["mean", "std", "min", "max"])

    # Daily volatility (by day of the week)
    daily_volatility = df.groupby("day_of_week")[country].agg(["mean", "std", "min", "max"])

    # Monthly volatility
    monthly_volatility = df.groupby("month")[country].agg(["mean", "std", "min", "max"])

    return volatility, hourly_volatility, daily_volatility, monthly_volatility

# Calculate for Spain and France
volatility_es, hourly_vol_es, daily_vol_es, monthly_vol_es = calculate_volatility(df, 'Spain')
volatility_fr, hourly_vol_fr, daily_vol_fr, monthly_vol_fr = calculate_volatility(df, 'France')

# --- 4. Visualizations ---
def plot_comparative_volatility(df):
    plt.figure(figsize=(18, 12))

    # 4.1. Time series of prices (Spain vs France)
    plt.subplot(2, 2, 1)
    df['Spain'].plot(label='Spain', color='blue', alpha=0.7)
    df['France'].plot(label='France', color='red', alpha=0.7)
    plt.title("Hourly Price Evolution (Spain vs France)")
    plt.ylabel("Price (€/MWh)")
    plt.legend()
    plt.grid(True)

    # 4.2. Price distribution (Spain vs France)
    plt.subplot(2, 2, 2)
    sns.histplot(df['Spain'], bins=30, kde=True, color='blue', label='Spain', alpha=0.5)
    sns.histplot(df['France'], bins=30, kde=True, color='red', label='France', alpha=0.5)
    plt.title("Price Distribution")
    plt.xlabel("Price (€/MWh)")
    plt.legend()

    # 4.3. Hourly volatility (standard deviation)
    plt.subplot(2, 2, 3)
    hourly_std = df.groupby("hour")[['Spain', 'France']].std()
    hourly_std.plot(kind='bar', title="Hourly Volatility (Standard Deviation)", alpha=0.7)
    plt.xlabel("Hour of the Day")
    plt.ylabel("Standard Deviation")

    # 4.4. Weekly volatility (standard deviation by day of the week)
    plt.subplot(2, 2, 4)
    daily_std = df.groupby("day_of_week")[['Spain', 'France']].std()
    days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    daily_std.plot(kind='bar', title="Volatility by Day of the Week", alpha=0.7)
    plt.xticks(range(7), days)
    plt.ylabel("Standard Deviation")

    plt.tight_layout()
    plt.show()

    # 4.5. Boxplot by month (Spain vs France)
    plt.figure(figsize=(12, 6))
    df_melted = df.reset_index().melt(id_vars=['hour', 'day_of_week', 'month', 'date'],
                                      value_vars=['Spain', 'France'],
                                      var_name='Country', value_name='Price')
    sns.boxplot(x="month", y="Price", hue="Country", data=df_melted)
    plt.title("Price Distribution by Month (Spain vs France)")
    plt.xlabel("Month")
    plt.ylabel("Price (€/MWh)")
    plt.grid(True)
    plt.show()

# --- 5. Display Results ---
print("--- Global Volatility (Spain) ---")
for k, v in volatility_es.items():
    print(f"{k}: {v:.2f}")

print("\n--- Global Volatility (France) ---")
for k, v in volatility_fr.items():
    print(f"{k}: {v:.2f}")

# Generate visualizations
plot_comparative_volatility(df)